In [1]:
import os
import itertools
import numpy as np
import pandas as pd
import anndata as ad
import umap

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, calinski_harabasz_score

# =========================
# 1. Load data
# =========================
input_dir = "/Users/apple/Desktop/KB/data"
adata_train = ad.read_h5ad(input_dir + '/LarryData/train_test/Larry_200_train.h5ad')
adata_test  = ad.read_h5ad(input_dir + '/LarryData/train_test/Larry_200_test.h5ad')

train_labels = adata_train.obs["clone_id"].to_numpy()
test_labels  = adata_test.obs["clone_id"].to_numpy()

train_data = adata_train.X
test_data  = adata_test.X

print(f"Data loaded: train {train_data.shape}, test {test_data.shape}")
print(f"Train labels: {train_labels.shape}, Test labels: {test_labels.shape}")

# =========================
# 2. Output locations
# =========================
save_dir = "/Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings"
os.makedirs(save_dir, exist_ok=True)

results_csv = os.path.join(save_dir, "supUMAP_hyperparameter_sweep_results.csv")

# =========================
# 3. Hyperparameter sweep
# =========================
param_grid = {
    "n_components": [10, 32, 50],
    "n_neighbors": [15, 30],
    "target_weight": [0.5, 0.8],
}

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

results = []

# =========================
# 4. Run sweep
# =========================
for idx, params in enumerate(combinations, start=1):
    print(f"\n--- Running config {idx}/{len(combinations)}: {params} ---")

    reducer = umap.UMAP(
        n_components=params["n_components"],
        n_neighbors=params["n_neighbors"],
        target_weight=params["target_weight"],
        random_state=42,
    )

    # Fit on train, transform train/test
    train_embeddings = reducer.fit_transform(train_data, y=train_labels)
    test_embeddings  = reducer.transform(test_data)

    # Save embeddings for later KL analysis
    name = f"nn{params['n_neighbors']}_dim{params['n_components']}_tw{params['target_weight']}"
    train_path = os.path.join(save_dir, f"train_{name}.npy")
    test_path  = os.path.join(save_dir, f"test_{name}.npy")

    np.save(train_path, train_embeddings)
    np.save(test_path, test_embeddings)

    # CH scores
    train_ch = calinski_harabasz_score(train_embeddings, train_labels)
    test_ch  = calinski_harabasz_score(test_embeddings, test_labels)

    # KNN test accuracy/error
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(train_embeddings, train_labels)
    y_pred_test = knn.predict(test_embeddings)

    test_acc = accuracy_score(test_labels, y_pred_test)
    test_err = 1 - test_acc

    results.append({
        "embedding_name": name,
        "n_components": params["n_components"],
        "n_neighbors": params["n_neighbors"],
        "target_weight": params["target_weight"],
        "Test_KNN_Acc": test_acc,
        "Test_KNN_Error": test_err,
        "Train_CH_Score": train_ch,
        "Test_CH_Score": test_ch,
        "train_embedding_path": train_path,
        "test_embedding_path": test_path,
    })

    print(f"Saved train embedding to: {train_path}")
    print(f"Saved test embedding to:  {test_path}")
    print(f"Test KNN accuracy: {test_acc:.4f}")
    print(f"Test KNN error:    {test_err:.4f}")
    print(f"Train CH score:    {train_ch:.4f}")
    print(f"Test CH score:     {test_ch:.4f}")

# =========================
# 5. Save and inspect results
# =========================
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="Test_KNN_Acc", ascending=False)

print("\n==== HYPERPARAMETER SWEEP RESULTS ====")
print(df_results.to_string(index=False))

df_results.to_csv(results_csv, index=False)
print(f"\nSaved summary results to: {results_csv}")

Data loaded: train (10148, 2000), test (1225, 2000)
Train labels: (10148,), Test labels: (1225,)

--- Running config 1/12: {'n_components': 10, 'n_neighbors': 15, 'target_weight': 0.5} ---


/opt/anaconda3/envs/lcl_clean/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved train embedding to: /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/train_nn15_dim10_tw0.5.npy
Saved test embedding to:  /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/test_nn15_dim10_tw0.5.npy
Test KNN accuracy: 0.0653
Test KNN error:    0.9347
Train CH score:    124.7656
Test CH score:     15.1669

--- Running config 2/12: {'n_components': 10, 'n_neighbors': 15, 'target_weight': 0.8} ---


/opt/anaconda3/envs/lcl_clean/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved train embedding to: /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/train_nn15_dim10_tw0.8.npy
Saved test embedding to:  /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/test_nn15_dim10_tw0.8.npy
Test KNN accuracy: 0.0686
Test KNN error:    0.9314
Train CH score:    119.9300
Test CH score:     15.1313

--- Running config 3/12: {'n_components': 10, 'n_neighbors': 30, 'target_weight': 0.5} ---


...

Saved train embedding to: /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/train_nn15_dim50_tw0.8.npy
Saved test embedding to:  /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/test_nn15_dim50_tw0.8.npy
Test KNN accuracy: 0.0718
Test KNN error:    0.9282
Train CH score:    114.9771
Test CH score:     14.9527

--- Running config 11/12: {'n_components': 50, 'n_neighbors': 30, 'target_weight': 0.5} ---


/opt/anaconda3/envs/lcl_clean/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved train embedding to: /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/train_nn30_dim50_tw0.5.npy
Saved test embedding to:  /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/test_nn30_dim50_tw0.5.npy
Test KNN accuracy: 0.0727
Test KNN error:    0.9273
Train CH score:    124.8223
Test CH score:     15.0804

--- Running config 12/12: {'n_components': 50, 'n_neighbors': 30, 'target_weight': 0.8} ---


/opt/anaconda3/envs/lcl_clean/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved train embedding to: /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/train_nn30_dim50_tw0.8.npy
Saved test embedding to:  /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/test_nn30_dim50_tw0.8.npy
Test KNN accuracy: 0.0718
Test KNN error:    0.9282
Train CH score:    111.7375
Test CH score:     13.9696

==== HYPERPARAMETER SWEEP RESULTS ====
  embedding_name  n_components  n_neighbors  target_weight  Test_KNN_Acc  Test_KNN_Error  Train_CH_Score  Test_CH_Score                                                             train_embedding_path                                                             test_embedding_path
nn15_dim32_tw0.8            32           15            0.8      0.073469        0.926531      117.546618      15.070643 /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/train_nn15_dim32_tw0.8.npy /Users/apple/Desktop/KB/data/supUMAP_sweep_embeddings/test_nn15_dim32_tw0.8.npy
nn30_dim50_tw0.5            50           30            0.5      0.072653      